# UX Landing Page Analysis with OpenAI Vision

This notebook demonstrates how to use pyppeteer to capture full-page screenshots of websites (desktop and mobile), then analyze those landing pages using the OpenAI Responses API with vision capabilities to produce structured UX feedback.

In [ ]:
%pip install pyppeteer pyppeteer_stealth openai pydantic --upgrade --quiet

In [ ]:
import asyncio
import base64
import glob
import json
import os
import textwrap
import urllib.parse
from getpass import getpass
from typing import List, Optional

from pyppeteer import launch
from pyppeteer_stealth import stealth
from openai import OpenAI
from pydantic import BaseModel, Field

MODEL = "gpt-4o-mini"

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI()

In [ ]:
# Setting custom user agents to help avoid detection:
MOBILE_USER_AGENT = "Mozilla/5.0 (Linux; Android 7.0; SM-G892A Build/NRD90M) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/59.0.3071.125 Mobile Safari/537.36"
DESKTOP_USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3"

# TODO - You must edit this!
DOMAIN = "understandingdata.com" # Change this to the domain you want to use as your base domain

In [ ]:
async def screenshot_full_page(url, screenshot_filename, device_type="desktop"):
    browser = await launch(headless=True)
    page = await browser.newPage()

    if device_type == "mobile":
        await page.emulate(
            {
                "viewport": {"width": 360, "height": 640, "isMobile": True},
                "hasTouch": True,
                "isMobile": True,
                "userAgent": MOBILE_USER_AGENT,
            }
        )
    else:
        await page.setViewport({"width": 1280, "height": 800})
        await page.emulate({
            "viewport": {"width": 1280, "height": 800},
            "userAgent": DESKTOP_USER_AGENT

        })

    await stealth(page)
    await page.goto(url, {"waitUntil": "networkidle0"})

    # Scroll to the bottom to ensure all lazy-loaded images are loaded:
    await page.evaluate("window.scrollTo(0, document.body.scrollHeight);")
    await page.waitFor(2000)  # Wait for lazy-loaded images

    # Scroll back to the top:
    await page.evaluate("window.scrollTo(0, 0);")

    # Take screenshot of the entire page, change the viewport to the full page:
    await page.setViewport(
        {
            "width": await page.evaluate("document.body.scrollWidth"),
            "height": await page.evaluate("document.body.scrollHeight"),
        }
    )

    # Take screenshot:
    screenshot_bytes = await page.screenshot()

    # Save the screenshot
    with open(screenshot_filename, "wb") as f:
        f.write(screenshot_bytes)
    await browser.close()

In [ ]:
# List of URLs to take screenshots of:
urls = [
    "https://understandingdata.com/",  # Replace with your main website URL
    "https://www.dufrain.co.uk/data-solutions/data-engineering/",  # Replace with the first competitor URL
    "https://www.fdmgroup.com/services/technical-services/data-engineering/",  # Replace with the second competitor URL
]

if DOMAIN not in urls[0]:
    raise ValueError(f"The first URL must be from the domain {DOMAIN}")

# Make clean names from the urls using urllib:
clean_names = [urllib.parse.urlparse(url).netloc for url in urls]

# Loop through the URLs and take screenshots
async def take_screenshot(url, clean_name, device_type):
    filename = f"screenshot_{clean_name}_{device_type}.png"
    print(f"Taking screenshot of {url} and saving to {filename}")
    await screenshot_full_page(url, filename, device_type)

async def main():
    tasks = []
    for url, clean_name in zip(urls, clean_names):
        for device_type in ["desktop", "mobile"]:
            tasks.append(take_screenshot(url, clean_name, device_type))
    await asyncio.gather(*tasks)

await main()

----------------------------------------------------------------------

## Making A Call To Vision API

In [ ]:
# Load all of the .png files in the current directory:
screenshot_files = glob.glob("*.png")

# Convert all of them to base64:
screenshot_base64s = {}
for filename in screenshot_files:
    with open(filename, "rb") as f:
        screenshot_base64s[filename] = base64.b64encode(f.read()).decode("utf-8")

In [ ]:
# Defining Pydantic models for structured output:
class FeedbackAspect(BaseModel):
    aspect: str
    description: str
    recommendations: Optional[List[str]] = None

class LandingPageFeedback(BaseModel):
    website_url: str
    strengths: List[FeedbackAspect]
    areas_for_improvement: List[FeedbackAspect]
    general_feedback: Optional[str] = None
    additional_comments: Optional[str] = None

In [ ]:
# Define the JSON schema inline to avoid $defs references from model_json_schema()
landing_page_schema = {
    "type": "object",
    "properties": {
        "website_url": {"type": "string"},
        "strengths": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "aspect": {"type": "string"},
                    "description": {"type": "string"},
                    "recommendations": {
                        "anyOf": [
                            {"type": "array", "items": {"type": "string"}},
                            {"type": "null"}
                        ]
                    }
                },
                "required": ["aspect", "description", "recommendations"],
                "additionalProperties": False
            }
        },
        "areas_for_improvement": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "aspect": {"type": "string"},
                    "description": {"type": "string"},
                    "recommendations": {
                        "anyOf": [
                            {"type": "array", "items": {"type": "string"}},
                            {"type": "null"}
                        ]
                    }
                },
                "required": ["aspect", "description", "recommendations"],
                "additionalProperties": False
            }
        },
        "general_feedback": {
            "anyOf": [{"type": "string"}, {"type": "null"}]
        },
        "additional_comments": {
            "anyOf": [{"type": "string"}, {"type": "null"}]
        }
    },
    "required": [
        "website_url",
        "strengths",
        "areas_for_improvement",
        "general_feedback",
        "additional_comments"
    ],
    "additionalProperties": False
}

system_content = f"""Act as a marketing user researcher.
You will receive a set of screenshots of your website and your different websites.
Your website is {urls[0]} and {clean_names[0]}.
---
The different websites are {urls[1]} and {clean_names[1]}, and {urls[2]} and {clean_names[2]}.
---
Please provide a brief analysis of the screenshots and identify any areas for improvement on your website.
You are allowed to use the different websites for research purposes.

Return your response as valid JSON matching this schema:
{json.dumps(landing_page_schema, indent=2)}
"""

In [ ]:
# Build image content for our website
our_images = [
    {"type": "input_image", "image_url": f"data:image/jpeg;base64,{screenshot_base64s[key]}"}
    for key in screenshot_base64s if DOMAIN in key
]

# Build image content for competitor websites
competitor_images = [
    {"type": "input_image", "image_url": f"data:image/jpeg;base64,{screenshot_base64s[key]}"}
    for key in screenshot_base64s if DOMAIN not in key
]

response = client.responses.create(
    model=MODEL,
    instructions=system_content,
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": f"My website is {urls[0]} and {clean_names[0]}. Please find attached both the mobile and desktop version."},
            ] + our_images,
        },
        {
            "role": "assistant",
            "content": "Thanks for providing your web pages in both desktop and mobile versions. Before analysing them, I will need to research the different websites to understand the competition. Can you provide some information on the different websites?",
        },
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": "Sure, here are some competitor images"},
            ] + competitor_images,
        },
    ],
    text={
        "format": {
            "type": "json_schema",
            "name": "landing_page_feedback",
            "strict": True,
            "schema": landing_page_schema,
        }
    },
    max_output_tokens=2000,
)

result = LandingPageFeedback.model_validate_json(response.output_text)
print(result)

In [ ]:
print(type(result))

In [ ]:
print(result.model_dump_json(indent=2))

---

## Edge Cases/Next Steps To Improve The Script:

1. Click on cookie banners to accept cookies, edge case: https://www.dufrain.co.uk/data-solutions/data-engineering/
2. Click on any pop ups such as banner ads, edge case: https://www.dufrain.co.uk/data-solutions/data-engineering/
3. Create an x,y cordinate grid that will allow us to completely control the vision model.

## Summary

- **pyppeteer** captures full-page screenshots (desktop and mobile) of your website and competitor sites, using stealth mode to avoid bot detection.
- Screenshots are base64-encoded and sent as vision inputs to the **OpenAI Responses API**.
- The API returns **structured JSON output** validated against a strict JSON schema, which is then parsed into a **Pydantic model** (`LandingPageFeedback`).
- This approach gives you actionable, structured UX feedback -- identifying strengths, areas for improvement, and concrete recommendations -- all driven by a single API call with vision capabilities.